# PAMAP2 Activity Recognition with LSTM

## Project Context

This notebook develops the wearable activity-recognition component of the **Integrated Intelligent Health Monitoring and Clinical Decision Support System**.

The goal is to classify physical activities from multivariate wearable sensor sequences using the PAMAP2 dataset.

The notebook uses reusable project modules:

- `src/preprocessing/pamap2.py` loads, cleans, windows, splits, and scales sensor data;
- `src/activity/lstm.py` defines the LSTM model and reusable training/evaluation functions.

The model is evaluated on subjects that are not present in the training set, providing a stronger test of generalization to unseen participants.


## Modeling Strategy

A compact single-layer Long Short-Term Memory (LSTM) network is used as the baseline model.

The architecture uses:

- input sequence length: 100 time steps;
- input features: 19 selected sensor channels;
- hidden size: 64;
- recurrent layers: 1;
- output classes: 12 activities;
- loss function: weighted cross-entropy;
- optimizer: Adam;
- learning rate: 0.001;
- batch size: 64;
- training epochs: 15.

Weighted class loss is used because the activity classes are not perfectly balanced.


## Setup

Import the required libraries, locate the project root, and initialize reproducibility settings.


In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

cwd = Path.cwd()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the healthcare_capstone project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", os.getcwd())
print("Project root:", PROJECT_ROOT)

RANDOM_STATE = 42
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 0.001
HIDDEN_SIZE = 64
NUM_LAYERS = 1


In [ ]:
# Path configuration is handled above.


## Import Project Modules

Import the reusable PAMAP2 preprocessing and LSTM functions.


In [ ]:
from src.preprocessing.pamap2 import (
    WINDOW_SIZE,
    SELECTED_SENSOR_FEATURES,
    prepare_pamap2_data,
    preprocessing_summary,
    split_and_scale_pamap2,
    split_summary,
)

from src.activity.lstm import (
    build_activity_model,
    compute_class_weights,
    create_dataloader,
    evaluate_activity_model,
    evaluation_to_dict,
    get_device,
    load_activity_model,
    predict_activity_probabilities,
    save_activity_model,
    set_random_seed,
    train_activity_model,
)


In [ ]:
set_random_seed(RANDOM_STATE)

device = get_device()

print("PyTorch version:", torch.__version__)
print("Device:", device)


## Prepare PAMAP2 Sequences

Load all subject files, remove transient activity periods, interpolate missing sensor values, construct activity-safe windows, encode the activity labels, and retain participant identifiers.


In [ ]:
PAMAP2_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pamap2"
)

prepared = prepare_pamap2_data(
    PAMAP2_DIR
)

preprocessing_summary(prepared)


In [ ]:
print("Sequence array shape:", prepared.X.shape)
print("Label array shape:", prepared.y.shape)
print("Group array shape:", prepared.groups.shape)

print("\nClasses:")
for index, class_name in enumerate(prepared.class_names):
    print(index, class_name)


## Subject-Aware Split and Scaling

The data are split by participant rather than by individual sequence windows. This prevents windows from the same person appearing in both training and testing data.

The scaler is fitted only on training sequences, then applied to the testing sequences.


In [ ]:
split = split_and_scale_pamap2(
    prepared,
    test_size=0.22,
    random_state=RANDOM_STATE,
)

split_summary(split)


In [ ]:
train_subjects = sorted(set(split.groups_train))
test_subjects = sorted(set(split.groups_test))
subject_overlap = set(train_subjects).intersection(test_subjects)

print("Training subjects:", train_subjects)
print("Testing subjects:", test_subjects)
print("Subject overlap:", subject_overlap)

assert len(subject_overlap) == 0


## Inspect Class Distribution

Review the training and testing class counts after subject-aware splitting.


In [ ]:
train_counts = pd.Series(
    split.y_train
).value_counts().sort_index()

test_counts = pd.Series(
    split.y_test
).value_counts().sort_index()

class_distribution = pd.DataFrame({
    "class_name": prepared.class_names,
    "train_windows": [
        int(train_counts.get(i, 0))
        for i in range(len(prepared.class_names))
    ],
    "test_windows": [
        int(test_counts.get(i, 0))
        for i in range(len(prepared.class_names))
    ],
})

class_distribution


In [ ]:
plt.figure(figsize=(12, 6))

x = np.arange(len(prepared.class_names))
width = 0.4

plt.bar(
    x - width / 2,
    class_distribution["train_windows"],
    width=width,
    label="Training"
)

plt.bar(
    x + width / 2,
    class_distribution["test_windows"],
    width=width,
    label="Testing"
)

plt.title("PAMAP2 Window Distribution by Activity")
plt.xlabel("Activity")
plt.ylabel("Number of Windows")
plt.xticks(
    x,
    prepared.class_names,
    rotation=60,
    ha="right"
)
plt.legend()

plt.tight_layout()
plt.show()


## Create PyTorch DataLoaders

Training data are shuffled each epoch. Testing data are kept in a fixed order.


In [ ]:
train_loader = create_dataloader(
    split.X_train,
    split.y_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = create_dataloader(
    split.X_test,
    split.y_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))


## Compute Class Weights

Inverse-frequency class weights reduce the dominance of more common activities during training.


In [ ]:
class_weights = compute_class_weights(
    split.y_train,
    num_classes=len(prepared.class_names),
)

class_weight_table = pd.DataFrame({
    "class_name": prepared.class_names,
    "class_weight": class_weights.numpy(),
})

class_weight_table


## Build the LSTM

The baseline model uses one recurrent layer with 64 hidden units. This keeps the architecture compact while preserving the ability to learn temporal dependencies across sensor windows.


In [ ]:
model = build_activity_model(
    input_size=split.X_train.shape[2],
    num_classes=len(prepared.class_names),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
)

model


In [ ]:
parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", parameter_count)
print("Trainable parameters:", trainable_parameter_count)


## Train the Model

Training and testing behavior are recorded across epochs so that convergence and potential overfitting can be inspected.


In [ ]:
model, history = train_activity_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    device=device,
    class_weights=class_weights,
    verbose=True,
)


## Training Loss

Compare training and validation loss across epochs.


In [ ]:
epochs_range = range(
    1,
    len(history.train_loss) + 1
)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_range,
    history.train_loss,
    marker="o",
    label="Training Loss",
)

plt.plot(
    epochs_range,
    history.val_loss,
    marker="o",
    label="Validation Loss",
)

plt.title("LSTM Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()

plt.tight_layout()
plt.show()


## Training Accuracy

Compare training and validation accuracy across epochs.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    epochs_range,
    history.train_accuracy,
    marker="o",
    label="Training Accuracy",
)

plt.plot(
    epochs_range,
    history.val_accuracy,
    marker="o",
    label="Validation Accuracy",
)

plt.title("LSTM Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


## Final Model Evaluation

Evaluate performance using:

- accuracy;
- weighted precision;
- weighted recall;
- weighted F1 score;
- confusion matrix.

Weighted multiclass metrics are used because activity classes have different numbers of windows.


In [ ]:
evaluation = evaluate_activity_model(
    model,
    test_loader,
    device=device,
)

evaluation_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
    ],
    "Value": [
        evaluation.accuracy,
        evaluation.precision,
        evaluation.recall,
        evaluation.f1,
    ]
})

evaluation_metrics


## Confusion Matrix

The confusion matrix shows which activity pairs are most frequently confused on unseen participants.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ConfusionMatrixDisplay(
    confusion_matrix=evaluation.confusion_matrix,
    display_labels=prepared.class_names,
).plot(
    ax=ax,
    xticks_rotation=60,
    values_format="d",
)

plt.title("PAMAP2 LSTM Confusion Matrix")
plt.tight_layout()
plt.show()


## Per-Class Classification Report

Per-class precision, recall, and F1 scores help identify activity classes that are harder to recognize than the overall weighted metrics suggest.


In [ ]:
report = classification_report(
    evaluation.y_true,
    evaluation.y_pred,
    labels=np.arange(len(prepared.class_names)),
    target_names=prepared.class_names,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T

report_df


## Best and Weakest Activity Classes

Rank activities by per-class F1 score.


In [ ]:
per_class_results = (
    report_df.loc[
        prepared.class_names,
        ["precision", "recall", "f1-score", "support"]
    ]
    .sort_values(
        "f1-score",
        ascending=False
    )
)

per_class_results


In [ ]:
plt.figure(figsize=(12, 6))

per_class_results["f1-score"].sort_values(
    ascending=False
).plot(kind="bar")

plt.title("Per-Class F1 Score")
plt.xlabel("Activity")
plt.ylabel("F1 Score")
plt.xticks(rotation=60, ha="right")

plt.tight_layout()
plt.show()


## Prediction Confidence

Generate probability outputs for a small sample of testing windows and compare the predicted activity with the true activity.


In [ ]:
sample_size = 10

sample_probabilities = predict_activity_probabilities(
    model,
    split.X_test[:sample_size],
    batch_size=BATCH_SIZE,
    device=device,
)

sample_predictions = sample_probabilities.argmax(axis=1)
sample_confidence = sample_probabilities.max(axis=1)

sample_results = pd.DataFrame({
    "true_activity": [
        prepared.class_names[index]
        for index in split.y_test[:sample_size]
    ],
    "predicted_activity": [
        prepared.class_names[index]
        for index in sample_predictions
    ],
    "confidence": sample_confidence,
})

sample_results


## Save the Trained Activity Model

Save the trained PyTorch state dictionary together with the feature names, class names, architecture metadata, and window size required for reconstruction during later inference.


In [ ]:
MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "activity_model.pt"
)

saved_model_path = save_activity_model(
    model,
    MODEL_PATH,
    feature_names=prepared.feature_names,
    class_names=prepared.class_names,
    window_size=WINDOW_SIZE,
)

print("Activity model saved to:", saved_model_path)


## Verify Model Reloading

Reload the saved model and confirm that it reproduces the same class probabilities for a sample of testing windows.


In [ ]:
loaded_model, model_metadata = load_activity_model(
    MODEL_PATH,
    device=device,
)

original_probabilities = predict_activity_probabilities(
    model,
    split.X_test[:20],
    device=device,
)

loaded_probabilities = predict_activity_probabilities(
    loaded_model,
    split.X_test[:20],
    device=device,
)

reload_match = np.allclose(
    original_probabilities,
    loaded_probabilities,
    atol=1e-6,
)

print("Reloaded model predictions match:", reload_match)
print("\nSaved model metadata:")
print(model_metadata)

assert reload_match


## Save Evaluation Outputs

Save metrics, per-class results, class distributions, and training history for later system evaluation and reporting.


In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pamap2"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

evaluation_metrics.to_csv(
    OUTPUT_DIR / "activity_model_metrics.csv",
    index=False,
)

per_class_results.to_csv(
    OUTPUT_DIR / "activity_model_per_class_metrics.csv"
)

class_distribution.to_csv(
    OUTPUT_DIR / "activity_model_class_distribution.csv",
    index=False,
)

history_df = pd.DataFrame({
    "epoch": list(epochs_range),
    "train_loss": history.train_loss,
    "validation_loss": history.val_loss,
    "train_accuracy": history.train_accuracy,
    "validation_accuracy": history.val_accuracy,
})

history_df.to_csv(
    OUTPUT_DIR / "activity_model_training_history.csv",
    index=False,
)

print("Activity-model outputs saved to:", OUTPUT_DIR)


## Modeling Summary

This notebook establishes the second predictive branch of the integrated capstone system.

The workflow:

1. loads and preprocesses the public PAMAP2 wearable dataset;
2. converts continuous sensor streams into fixed-length LSTM sequences;
3. separates subjects between training and testing data;
4. scales sensor values using training data only;
5. trains a compact single-layer LSTM;
6. evaluates generalization on unseen participants;
7. analyzes activity-specific errors through a confusion matrix and per-class metrics;
8. saves and reloads the trained model successfully.

The final measured Accuracy, weighted Precision, weighted Recall, and weighted F1 score should be used in the integrated system evaluation and reflective synthesis paper.


## Next Step

After the wearable branch is complete, the project can proceed to integration.

The next implementation area is the **generative explanation and agentic orchestration layer**, which will combine structured outputs from:

- the diabetes readmission model;
- the PAMAP2 activity-recognition model;

while preserving their separate data provenance and maintaining non-diagnostic decision-support boundaries.
